# Notebook 17 — Final Ensemble + Submission

**Goal:** Combine MLP (NB15) + LGBM NB12 re-tuned via rank average to generate the best submission.

**Strategy confirmed in NB15:**
- Rank avg MLP+LGBM = **0.9150** CV AUC (+0.0118 vs previous best 0.9032)
- Projected LB: ~**0.940** (using confirmed +0.025 CV→LB boost)
- 2-model beats 3-model — CatBoost (0.9016) is weakest and dilutes rank ordering
- MLP ρ=0.791 vs LGBM — genuine architectural diversity that breaks the GBM ceiling

**Inputs:**
| File | Column | CV AUC |
|------|--------|--------|
| `oof_predictions_nb12.parquet` | `lgbm_tuned_oof` | 0.9024 |
| `oof_predictions_nb15.parquet` | `mlp_oof` | 0.9102 |
| `test_predictions_nb12.parquet` | `lgbm_tuned_pred` | — |
| `test_predictions_nb15.parquet` | `mlp_pred` | — |

**Output:** `submissions/submission_v004_mlp_lgbm_rank_avg.csv`

In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

# Path detection — works in VSCode and on Kaggle
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root = cwd
processed_dir  = project_root / 'data' / 'processed'
submissions_dir = project_root / 'submissions'
models_dir     = project_root / 'models'

print(f'Project root : {project_root}')
print(f'Processed    : {processed_dir.exists()}')
print(f'Submissions  : {submissions_dir.exists()}')

Project root : c:\Repos\predict-f1-pit-stops
Processed    : True
Submissions  : True


In [2]:
# Load OOF predictions from NB12 (LGBM re-tuned), NB15 (MLP), NB11 (CB plain)
oof12 = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')
oof15 = pd.read_parquet(processed_dir / 'oof_predictions_nb15.parquet')
oof11 = pd.read_parquet(processed_dir / 'oof_predictions_nb11.parquet')

print('NB12 columns:', oof12.columns.tolist())
print('NB15 columns:', oof15.columns.tolist())
print('NB11 columns:', oof11.columns.tolist())

# Verify individual AUCs match expectations
auc_lgbm = roc_auc_score(oof12['PitNextLap'], oof12['lgbm_tuned_oof'])
auc_mlp  = roc_auc_score(oof15['PitNextLap'], oof15['mlp_oof'])
auc_cb   = roc_auc_score(oof11['PitNextLap'], oof11['cb_plain_oof'])

print(f'\nVerification (expected → actual):')
print(f'  LGBM NB12 : 0.9024 → {auc_lgbm:.4f}')
print(f'  MLP  NB15 : 0.9102 → {auc_mlp:.4f}')
print(f'  CB   NB11 : 0.9016 → {auc_cb:.4f}')

NB12 columns: ['id', 'fold', 'PitNextLap', 'lgbm_tuned_oof', 'cb_tuned_oof']
NB15 columns: ['id', 'fold', 'PitNextLap', 'mlp_oof']
NB11 columns: ['id', 'fold', 'PitNextLap', 'lgbm_ref_oof', 'rf_oof', 'et_oof', 'dart_oof', 'cb_plain_oof']

Verification (expected → actual):
  LGBM NB12 : 0.9024 → 0.9024
  MLP  NB15 : 0.9102 → 0.9102
  CB   NB11 : 0.9016 → 0.9016


In [3]:
def rank_average(arrays):
    """Rank-average a list of prediction arrays. Returns normalized ranks in (0, 1)."""
    n = len(arrays[0])
    ranks = np.stack([rankdata(a) for a in arrays])  # shape: (n_models, n_rows)
    return ranks.mean(axis=0) / (n + 1)


# Merge all OOF predictions on id
oof = oof12[['id', 'fold', 'PitNextLap', 'lgbm_tuned_oof']].copy()
oof = oof.merge(oof15[['id', 'mlp_oof']], on='id')
oof = oof.merge(oof11[['id', 'cb_plain_oof']], on='id')
print(f'OOF merged shape: {oof.shape}')

# Spearman correlations (confirm NB15 values)
rho_mlp_lgbm, _ = spearmanr(oof['mlp_oof'], oof['lgbm_tuned_oof'])
rho_mlp_cb,   _ = spearmanr(oof['mlp_oof'], oof['cb_plain_oof'])
rho_lgbm_cb,  _ = spearmanr(oof['lgbm_tuned_oof'], oof['cb_plain_oof'])

print(f'\nSpearman \u03c1 (confirmation from NB15):')
print(f'  MLP \u00d7 LGBM : {rho_mlp_lgbm:.3f}  (expected 0.791)')
print(f'  MLP \u00d7 CB   : {rho_mlp_cb:.3f}  (expected 0.809)')
print(f'  LGBM \u00d7 CB  : {rho_lgbm_cb:.3f}  (expected 0.969)')

# All ensemble combinations — pick best
combos = [
    ('LGBM solo',    ['lgbm_tuned_oof'],                        'NB12 reference'),
    ('MLP solo',     ['mlp_oof'],                               'NB15 reference'),
    ('CB solo',      ['cb_plain_oof'],                          'NB11 reference'),
    ('LGBM+CB',      ['lgbm_tuned_oof', 'cb_plain_oof'],        'v002 baseline'),
    ('MLP+CB',       ['mlp_oof', 'cb_plain_oof'],               '2-model'),
    ('MLP+LGBM',     ['mlp_oof', 'lgbm_tuned_oof'],            '2-model OPTIMAL \u2605'),
    ('MLP+LGBM+CB',  ['mlp_oof', 'lgbm_tuned_oof', 'cb_plain_oof'], '3-model'),
]

print(f'\n{"Combination":<20} {"CV AUC":>8}  {"\u0394 v002":>8}  Note')
print('-' * 62)

results = {}
for name, cols, note in combos:
    preds = [oof[c].values for c in cols]
    avg = rank_average(preds) if len(preds) > 1 else preds[0]
    auc = roc_auc_score(oof['PitNextLap'], avg)
    delta = auc - 0.9032
    results[name] = {'auc': auc, 'cols': cols}
    print(f'  {name:<18} {auc:.4f}   {delta:+.4f}   {note}')

chosen_auc = results['MLP+LGBM']['auc']
print(f'\nChosen: MLP+LGBM  (CV AUC = {chosen_auc:.4f}, proj LB ~{chosen_auc + 0.025:.3f})')

OOF merged shape: (439140, 6)

Spearman ρ (confirmation from NB15):
  MLP × LGBM : 0.791  (expected 0.791)
  MLP × CB   : 0.809  (expected 0.809)
  LGBM × CB  : 0.969  (expected 0.969)

Combination            CV AUC    Δ v002  Note
--------------------------------------------------------------
  LGBM solo          0.9024   -0.0008   NB12 reference
  MLP solo           0.9102   +0.0070   NB15 reference
  CB solo            0.9016   -0.0016   NB11 reference
  LGBM+CB            0.9032   -0.0000   v002 baseline
  MLP+CB             0.9145   +0.0113   2-model
  MLP+LGBM           0.9150   +0.0118   2-model OPTIMAL ★
  MLP+LGBM+CB        0.9134   +0.0102   3-model

Chosen: MLP+LGBM  (CV AUC = 0.9150, proj LB ~0.940)


In [4]:
# Load test predictions
test12 = pd.read_parquet(processed_dir / 'test_predictions_nb12.parquet')
test15 = pd.read_parquet(processed_dir / 'test_predictions_nb15.parquet')

print('NB12 test cols:', test12.columns.tolist(), '| shape:', test12.shape)
print('NB15 test cols:', test15.columns.tolist(), '| shape:', test15.shape)

# Merge on id
test = test12[['id', 'lgbm_tuned_pred']].merge(
    test15[['id', 'mlp_pred']], on='id'
)
print(f'\nMerged test shape: {test.shape}')
print(f'ID range: {test["id"].min()} \u2013 {test["id"].max()}')

assert test.shape[0] == test12.shape[0] == test15.shape[0], 'ID count mismatch!'
assert test['id'].duplicated().sum() == 0, 'Duplicate IDs in test!'
print('\u2713 ID alignment OK')

NB12 test cols: ['id', 'lgbm_tuned_pred', 'cb_tuned_pred'] | shape: (188165, 3)
NB15 test cols: ['id', 'mlp_pred'] | shape: (188165, 2)

Merged test shape: (188165, 3)
ID range: 439140 – 627304
✓ ID alignment OK


In [5]:
# Rank average: MLP + LGBM (confirmed optimal from NB15)
test['PitNextLap'] = rank_average([
    test['mlp_pred'].values,
    test['lgbm_tuned_pred'].values,
])

# Build submission: id + PitNextLap, sorted by id
submission = test[['id', 'PitNextLap']].sort_values('id').reset_index(drop=True)

# Sanity checks
assert submission['id'].duplicated().sum() == 0, 'Duplicate IDs!'
assert submission['PitNextLap'].isnull().sum() == 0, 'NaN predictions!'
assert (submission['PitNextLap'] > 0).all() and (submission['PitNextLap'] < 1).all(), \
    'Predictions must be in (0, 1)!'

print('Submission shape:', submission.shape)
print(submission['PitNextLap'].describe().round(6).to_string())
print(f'\nFirst 5 rows:')
print(submission.head().to_string(index=False))

# Save
submission_path = submissions_dir / 'submission_v004_mlp_lgbm_rank_avg.csv'
submission.to_csv(submission_path, index=False)
print(f'\n\u2713 Saved: {submission_path}')
print(f'  File size: {submission_path.stat().st_size / 1e6:.2f} MB')

Submission shape: (188165, 2)
count    188165.000000
mean          0.500000
std           0.280453
min           0.000173
25%           0.259130
50%           0.482120
75%           0.748398
max           0.999944

First 5 rows:
    id  PitNextLap
439140    0.173087
439141    0.247364
439142    0.366806
439143    0.666914
439144    0.855845

✓ Saved: c:\Repos\predict-f1-pit-stops\submissions\submission_v004_mlp_lgbm_rank_avg.csv
  File size: 5.13 MB


In [6]:
# Save summary pickle for reproducibility
summary = {
    'ensemble_strategy': 'rank_avg_mlp_lgbm',
    'models': {
        'mlp':  {'source': 'oof_predictions_nb15.parquet', 'col': 'mlp_oof',  'cv_auc': auc_mlp},
        'lgbm': {'source': 'oof_predictions_nb12.parquet', 'col': 'lgbm_tuned_oof', 'cv_auc': auc_lgbm},
        'cb':   {'source': 'oof_predictions_nb11.parquet', 'col': 'cb_plain_oof', 'cv_auc': auc_cb},
    },
    'spearman_rho': {
        'mlp_lgbm': rho_mlp_lgbm,
        'mlp_cb':   rho_mlp_cb,
        'lgbm_cb':  rho_lgbm_cb,
    },
    'ensemble_results': {k: v['auc'] for k, v in results.items()},
    'chosen_combo': 'MLP+LGBM',
    'chosen_cv_auc': chosen_auc,
    'projected_lb': round(chosen_auc + 0.025, 4),
    'submission_file': submission_path.name,
    'submission_rows': len(submission),
}
with open(models_dir / 'nb17_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)
print('\u2713 Saved models/nb17_summary.pkl')

print()
print('=' * 62)
print('NB17 — Final Ensemble Summary')
print('=' * 62)
print(f'  Ensemble        : MLP NB15 + LGBM NB12 (rank average)')
print(f'  CV AUC          : {chosen_auc:.4f}')
print(f'  Projected LB    : ~{chosen_auc + 0.025:.3f}  (+0.025 boost)')
print(f'  Best LB so far  : 0.92845  (v002 LGBM+CB)')
print(f'  Expected gain   : ~{chosen_auc + 0.025 - 0.92845:+.3f} LB points')
print(f'  LB top          : 0.95488')
print(f'  Remaining gap   : ~{0.95488 - (chosen_auc + 0.025):.3f}')
print(f'  Submission file : {submission_path.name}')
print()
print('Add to leaderboard_log.md after uploading:')
print(f'| {submission_path.name} | rank avg MLP NB15 + LGBM NB12 | {chosen_auc:.4f} | [LB] | Submitted 2026-05-21 |')
print()
print('Next steps:')
print('  1. Upload submission_v004_mlp_lgbm_rank_avg.csv to Kaggle')
print('  2. Record LB AUC in submissions/leaderboard_log.md')
print('  3. If LB >= 0.935 → consider NB16 pseudo-labeling')
print('  4. If LB <  0.930 → investigate train/test distribution shift')

✓ Saved models/nb17_summary.pkl

NB17 — Final Ensemble Summary
  Ensemble        : MLP NB15 + LGBM NB12 (rank average)
  CV AUC          : 0.9150
  Projected LB    : ~0.940  (+0.025 boost)
  Best LB so far  : 0.92845  (v002 LGBM+CB)
  Expected gain   : ~+0.012 LB points
  LB top          : 0.95488
  Remaining gap   : ~0.015
  Submission file : submission_v004_mlp_lgbm_rank_avg.csv

Add to leaderboard_log.md after uploading:
| submission_v004_mlp_lgbm_rank_avg.csv | rank avg MLP NB15 + LGBM NB12 | 0.9150 | [LB] | Submitted 2026-05-21 |

Next steps:
  1. Upload submission_v004_mlp_lgbm_rank_avg.csv to Kaggle
  2. Record LB AUC in submissions/leaderboard_log.md
  3. If LB >= 0.935 → consider NB16 pseudo-labeling
  4. If LB <  0.930 → investigate train/test distribution shift


## NB17 Complete

**Submission:** `submission_v004_mlp_lgbm_rank_avg.csv`

**Expected LB AUC:** ~0.940 (CV 0.9150 + confirmed +0.025 boost)

### Decision gates after seeing LB result
| LB AUC | Action |
|--------|--------|
| ≥ 0.935 | Proceed to NB16 (pseudo-labeling) — further gains possible |
| 0.930–0.935 | Investigate whether a 3-model ensemble (MLP+LGBM+CB) helps on LB vs CV |
| < 0.930 | Investigate train/test distribution shift before further work |